# Inference the results of the thrmodynamic experiment

In [1]:
import sys
import gc
sys.path.append('../')

In [4]:
from model.data_loader import fetch_dataloader,fetch_inference_loader
from model.data_loader import params as data_params
from model.model_cfg import CFG
# from model.net import ProteinEnergyNet
from model.hydro_net import PEM
from model.net import params as model_params
from train_utils import *
import torch
import torch.nn.functional as F
from torch import optim
from torch.optim import lr_scheduler
from tqdm import tqdm
import gc
import time
import sys
import pandas as pd
import wandb

/home/shaharax/.conda/envs/esm2_env/lib/python3.7/site-packages/tqdm/auto.py:22: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [5]:
os.chdir("../")
os.getcwd()

'/casp15/Shahar/DeepPEF'

# set validation function

In [6]:
def criterion(Ejf, Ekf, Eju, Eku):
    """
    The loss function for the model coressponds to 2 main losses:
    1. lossg: delta energy betweeen the folded and unfolded structures
    2. lossd: the thermodynamic cycle loss
    Args:
        Ejf (tensor): The energy of the folded structure
        Ekf (tensor): The energy of the folded structure with mutation
        Eju (tensor): The energy of the unfolded structure
        Eku (tensor): The energy of the unfolded structure with mutation
    output:
        loss (tensor): The loss of the model
    """
    softplus = torch.nn.Softplus(beta=0.2)
    Ejf, Ekf, Eju, Eku = torch.tanh(Ejf), torch.tanh(Ekf), torch.tanh(Eju), torch.tanh(Eku) # clip the energy to be between -1 and 1
    delta_g1, delta_g2, delta_g3, delta_g4 = Ejf-Eju, Ekf-Ejf, Eku-Eju, Ekf-Eku # themodynamic cycle, from the paper
    lossg = ((delta_g1+delta_g2)-(delta_g3+delta_g4))**2
    # lossd = torch.log((torch.exp(Ekf-Eku) + torch.exp(Ejf-Eju)) +1)
    lossd = softplus(Ekf-Eku) + softplus(Ejf-Eju)
    return lossd+lossg , lossd, lossg  

In [7]:
# define validation function
def validation(model, dataloader, device,epoch,N,optimizer,val_type = 'robust'):
    """
    Validation function for the model.
    """
    valid_loss = 0
    Exd_list = []
    Exn_list = []
    seq_len = []
    lossg_list = []
    lossd_list = []
    ids_list = []
    n_skips = 0
    running_loss = 0.0
    model.eval() 
    with tqdm(dataloader, unit="batch") as tepoch:
        for index, data in (enumerate(tepoch)):
            # set progress bar description
            tepoch.set_description(f"Validation: Epoch {epoch}, running loss: {round(valid_loss/(index + 1),3)}")
            # Clean the GPU cache
            if(device.type == "cuda" or device.type == "mps"):    
                torch.cuda.empty_cache()
            gc.collect()
            # get the inputs; data is a list of [inputs, labels]   
            id, crd_backbone, mask, seq_one_hot, seq,ang_backbone, ang, proT5_emb, proT5_mut,seq_mut = data
            
            Xjf = crd_backbone.to(device) # wilde type structure folded
            Xkf = torch.clone(Xjf).to(device) # mutant structure folded
            Xju = torch.clone(Xjf).to(device) # wilde type structure unfolded
            Xku = torch.clone(Xjf).to(device) # mutant structure unfolded

            mask = mask.to(device)
            mask_decoy = torch.clone(mask).to(device)
            
            seq_one_hot = seq_one_hot.to(device) # [batch_size,seq_len,20]
            seq_one_hot_mut = get_one_hot(seq_mut[0]).to(device) # [batch_size,seq_len,20]
            
            if seq_one_hot.shape[1] >CFG.seq_len : # if the sequence is too long, skip it(GPU limitation)
                n_skips += 1
                continue
            
            emb = seq_one_hot.to(device)
            emb_decoy = seq_one_hot_mut.to(device)
            # move proT5_emb to device
            proT5_mut, proT5_emb = proT5_mut.to(device), proT5_emb.to(device)
            # zero the parameter gradients
            optimizer.zero_grad()
            # squeeze the data
            Xjf, Xkf, Xju, Xku = Xjf.squeeze(), Xkf.squeeze(), Xju.squeeze(), Xku.squeeze()
            emb_decoy, emb = emb_decoy.squeeze(), emb.squeeze()
            mask_decoy, mask= mask_decoy.squeeze(), mask.squeeze()
            proT5_mut, proT5_emb = proT5_mut.squeeze(), proT5_emb.squeeze()
            # get folded graph  
            Xjf,Xkf = get_graph(Xjf, emb, proT5_emb, mask), get_graph(Xkf, emb, proT5_mut, mask)
            # get unfolded graph
            Xju,Xku = get_unfolded_graph(Xju, emb, proT5_emb, mask), get_unfolded_graph(Xku, emb, proT5_mut, mask)
            # calculate the energy for the folded unfolded structures
            Ejf, Ekf, Eju, Eku = model(Xjf), model(Xkf), model(Xju), model(Xku)
            
            loss ,lossd, lossg = criterion(Ejf, Ekf, Eju, Eku)
            valid_loss += loss.item() 
            running_loss += loss.item()
            torch.cuda.empty_cache()
            gc.collect()
            # update the progress bar
            if index % 1000 == 999:
                print(f"Validation loss: {round(valid_loss/(index + 1),2)}, index: {index}, n_skips: {n_skips}")
                # validation_plots(Exd_list,Exn_list,seq_len,val_type,epoch)
            #tepoch.set_postfix({"loss":round(loss.item(),3),"running loss":round(valid_loss/(index + 1),3),"lossd":round(lossd.item(),3),"lossg":round(lossg.item(),3),"Exn":round(Exn.item(),3),"Exd":round(Exd.item(),3)})
            
            # Exd_list.append(Exd.item())
            # Exn_list.append(Exn.item())
            # seq_len.append(Xd.shape[0])
            lossg_list.append(lossg.item())
            lossd_list.append(lossd.item())
            ids_list.append(id)
    
    # validation_plots(Exd_list,Exn_list,seq_len,val_type,epoch)
    # df = pd.DataFrame({'id':ids_list,'Exd':Exd_list,'Exn':Exn_list,'seq_len':seq_len,'lossg':lossg_list,'lossd':lossd_list})
    # df.to_csv(f'./res/results/epoch_{epoch}-validation_{val_type}.csv')
    # print(f"Finished Validation {val_type} epoch {epoch}")
            
    return valid_loss/len(dataloader)


In [8]:
d_params = data_params(num_workers =CFG.num_workers, batch_size=CFG.batch_size,cuda=CFG.cuda,constraint=CFG.constraint, debug=CFG.debug,dataset='scn')
train_loader, valid_loader,test_loader = fetch_dataloader(data_dir=CFG.data_path, params=d_params)
print('***Build the model***')
model = PEM(dim_in=36,dim_h=64,dim_out=36,layers=CFG.num_layers,gaussian_coef=CFG.gaussian_coef).to(CFG.device)
model.name = "PEM-thermodynamic cycle"
optimizer = optim.Adam(model.parameters(), lr=CFG.lr, weight_decay=CFG.wd)

***Build the model***


In [9]:
load_checkpoint(CFG.model_path+f"best_model.pt", model, optimizer,CFG.device)
validation(model, valid_loader,CFG.device,-1, CFG.N, optimizer , val_type = 'robust')

Loaded model from ./res/trianed_models_th/33_final_model.pt


Validation: Epoch -1, running loss: 73.051:  46%|████▋     | 74/160 [00:25<00:29,  2.94batch/s] 


KeyboardInterrupt: 

In [ ]:
def get_exmample(data_loder_item, device):
    # get the inputs; data is a list of [inputs, labels]   
    id, crd_backbone, mask, seq_one_hot, seq,ang_backbone, ang, proT5_emb, proT5_mut,seq_mut = data_loder_item

    Xjf = crd_backbone.to(device) # wilde type structure folded
    Xkf = torch.clone(Xjf).to(device) # mutant structure folded
    Xju = torch.clone(Xjf).to(device) # wilde type structure unfolded
    Xku = torch.clone(Xjf).to(device) # mutant structure unfolded

    mask = mask.to(device)
    mask_decoy = torch.clone(mask).to(device)

    seq_one_hot = seq_one_hot.to(device) # [batch_size,seq_len,20]
    seq_one_hot_mut = get_one_hot(seq_mut[0]).to(device) # [batch_size,seq_len,20]

    emb = seq_one_hot.to(device)
    emb_decoy = seq_one_hot_mut.to(device)
    # move proT5_emb to device
    proT5_mut, proT5_emb = proT5_mut.to(device), proT5_emb.to(device)
    # squeeze the data
    Xjf, Xkf, Xju, Xku = Xjf.squeeze(), Xkf.squeeze(), Xju.squeeze(), Xku.squeeze()
    emb_decoy, emb = emb_decoy.squeeze(), emb.squeeze()
    mask_decoy, mask= mask_decoy.squeeze(), mask.squeeze()
    proT5_mut, proT5_emb = proT5_mut.squeeze(), proT5_emb.squeeze()
    # get folded graph  
    Xjf,Xkf = get_graph(Xjf, emb, proT5_emb, mask), get_graph(Xkf, emb, proT5_mut, mask)
    # get unfolded graph
    Xju,Xku = get_unfolded_graph(Xju, emb, proT5_emb, mask), get_unfolded_graph(Xku, emb, proT5_mut, mask)

    return Xjf,Xkf,Xju,Xku

In [ ]:
for item in valid_loader:
    Xjf,Xkf,Xju,Xku = get_exmample(item,CFG.device)
    print(Xjf.shape,Xkf.shape,Xju.shape,Xku.shape)
    Ejf, Ekf, Eju, Eku = model(Xjf), model(Xkf), model(Xju), model(Xku)
    print(Ejf.item(), Ekf.item(), Eju.item(), Eku.item())
    print(criterion(Ejf, Ekf, Eju, Eku))
    break

torch.Size([76, 1092]) torch.Size([76, 1092]) torch.Size([76, 1092]) torch.Size([76, 1092])
337.4724426269531 336.13299560546875 95.79502868652344 95.33118438720703
(tensor(inf, device='cuda:0', grad_fn=<AddBackward0>), tensor(inf, device='cuda:0', grad_fn=<LogBackward0>), tensor(2.3283e-10, device='cuda:0', grad_fn=<PowBackward0>))
